# Laboratorio 04 — Vistas y Optimización SQL sobre tu propio dataset

**Semana:** 03 | **Actividad de referencia:** Actividad 04  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica los conceptos de `CREATE OR REPLACE VIEW`, `EXPLAIN` y equivalencias SQL ↔ PySpark de la Actividad 04 sobre tu dataset personal. El objetivo es aprender a organizar consultas reutilizables y a leer el plan de ejecución para identificar cuellos de botella.

## Parte 1 — Descripción del dataset

1. **Nombre, fuente y URL** del dataset.
2. **Vista que crearás:** ¿Qué consulta de negocio encapsulará la vista? ¿Por qué tiene sentido reutilizarla?
3. **Preguntas de negocio** que responderás consultando la vista (no la tabla base directamente).

**Escribe tu respuesta aquí:**

## Parte 2 — Cargar el dataset como tabla Delta

In [ ]:
VOL     = "/Volumes/workspace/default/week_3"  # ajusta si usas otra ubicación
ARCHIVO = "tu_archivo.csv"
TABLA   = "workspace.default.lab03_04_mi_dataset"

df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(f"{VOL}/{ARCHIVO}")

df.write.format("delta").mode("overwrite").saveAsTable(TABLA)
print(f"✓ {TABLA}: {df.count():,} filas x {len(df.columns)} columnas")

## Parte 3 — Perfil técnico

In [ ]:
spark.sql(f"DESCRIBE TABLE EXTENDED {TABLA}").show(30, truncate=False)

In [ ]:
# Historial Delta de la tabla
spark.sql(f"DESCRIBE HISTORY {TABLA}").show(5, truncate=False)

## Parte 4 — Crear Vistas SQL

Crea al menos 2 vistas que encapsulen lógica reutilizable de tu dataset.

In [ ]:
# Vista 1: resumen de negocio base
# Define qué lógica encapsula esta vista y por qué sería útil reutilizarla
spark.sql(f"""
    CREATE OR REPLACE VIEW workspace.default.v_lab03_resumen AS
    SELECT
        columna_categoria,
        COUNT(*)              AS total_registros,
        AVG(columna_numerica) AS promedio,
        SUM(columna_numerica) AS total
    FROM {TABLA}
    WHERE columna_clave IS NOT NULL
    GROUP BY columna_categoria
""")
print("✓ Vista v_lab03_resumen creada")

In [ ]:
# Consultar la vista
spark.sql("SELECT * FROM workspace.default.v_lab03_resumen ORDER BY total DESC LIMIT 15").show(truncate=False)

In [ ]:
# Vista 2: vista analítica más compleja (usa window function o JOIN si aplica)
spark.sql(f"""
    CREATE OR REPLACE VIEW workspace.default.v_lab03_analitica AS
    SELECT
        *,
        RANK() OVER (
            PARTITION BY columna_categoria
            ORDER BY columna_numerica DESC
        ) AS ranking
    FROM {TABLA}
""")
print("✓ Vista v_lab03_analitica creada")

In [ ]:
# Consultar la vista analítica — solo top 1 por categoría
spark.sql("""
    SELECT *
    FROM workspace.default.v_lab03_analitica
    WHERE ranking = 1
    ORDER BY columna_categoria
""").show(truncate=False)

**Observaciones sobre las vistas:** ¿Las vistas almacenan datos o solo la definición de la consulta? ¿Qué pasa si modificas la tabla base después de crear la vista?

## Parte 5 — EXPLAIN: Analizar el plan de ejecución

In [ ]:
# Plan de ejecución de la consulta directa a la tabla
spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT columna_categoria, COUNT(*), AVG(columna_numerica)
    FROM {TABLA}
    GROUP BY columna_categoria
    ORDER BY 2 DESC
""").show(100, truncate=False)

In [ ]:
# Plan de ejecución de consulta sobre la vista
spark.sql("""
    EXPLAIN FORMATTED
    SELECT * FROM workspace.default.v_lab03_resumen WHERE total > 10
""").show(100, truncate=False)

**Análisis del plan:**
1. ¿Ves algún `FileScan` en el plan? ¿Cuántos archivos escanea?
2. ¿Hay algún `Exchange` (shuffle)? ¿En qué parte del plan aparece y por qué?
3. ¿El plan de la consulta sobre la vista es igual al de la consulta directa? ¿Por qué?  
4. ¿Qué optimización de Spark Catalyst puedes identificar en el plan?

## Parte 6 — Equivalencias SQL ↔ PySpark

Para cada operación SQL escribe su equivalente exacto en PySpark y viceversa.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Equivalente PySpark de: SELECT columna_categoria, COUNT(*), AVG(columna_numerica)
#                          FROM tabla GROUP BY columna_categoria ORDER BY 2 DESC
df = spark.table(TABLA)

df_equiv = df.groupBy("columna_categoria") \
    .agg(
        F.count("*").alias("total_registros"),
        F.avg("columna_numerica").alias("promedio")
    ) \
    .orderBy(F.col("total_registros").desc())

df_equiv.show(15, truncate=False)

In [ ]:
# Equivalente PySpark de la window function SQL
# SQL: RANK() OVER (PARTITION BY columna_categoria ORDER BY columna_numerica DESC)
w = Window.partitionBy("columna_categoria").orderBy(F.col("columna_numerica").desc())

df.withColumn("ranking", F.rank().over(w)) \
    .filter(F.col("ranking") == 1) \
    .orderBy("columna_categoria") \
    .show(truncate=False)

**Tabla comparativa SQL ↔ PySpark:**

| Concepto SQL | Equivalente PySpark |
|---|---|
| `WHERE col IS NOT NULL` | `.filter(F.col('col').isNotNull())` |
| `GROUP BY col` | `.groupBy('col')` |
| `HAVING COUNT(*) > N` | `.filter(F.col('count') > N)` |
| `ORDER BY col DESC` | `.orderBy(F.col('col').desc())` |
| `CREATE OR REPLACE VIEW v AS ...` | `df.createOrReplaceTempView('v')` |
| `RANK() OVER (PARTITION BY ...)` | `F.rank().over(Window.partitionBy(...))` |
| Añade más filas | para tu dataset |

## Parte 7 — Preguntas de negocio consultando las vistas

In [ ]:
# Pregunta 1 — usando v_lab03_resumen
spark.sql("""
    SELECT *
    FROM workspace.default.v_lab03_resumen
    -- añade tus condiciones
    LIMIT 10
""").show(truncate=False)

**Conclusión pregunta 1:**

In [ ]:
# Pregunta 2 — usando v_lab03_analitica
spark.sql("""
    SELECT *
    FROM workspace.default.v_lab03_analitica
    -- añade tus condiciones
    LIMIT 10
""").show(truncate=False)

**Conclusión pregunta 2:**

## Parte 8 — Reflexión final

1. ¿Cuándo usarías una vista en lugar de una tabla materializada (CTAS)?
2. ¿Qué encontraste en el plan `EXPLAIN` que no esperabas?
3. ¿Cuándo preferirías SQL puro sobre PySpark para escribir transformaciones en producción?
4. Si tuvieras que optimizar tu consulta más lenta, ¿qué harías basándote en el plan `EXPLAIN`?

---

## Entrega en Git

```bash
git add semana_03/laboratorios/lab_04_vistas_optimizacion.ipynb
git commit -m "lab: semana03 lab04 vistas EXPLAIN SQL-PySpark <nombre-dataset> - <tu-nombre>"
git push origin feature/semana03-sql-<tu-nombre>
```